# BDAT05 · MIDTERM · Part 2A — Practical Notebook A
**Panaderia Buena Masa** — a Laguna bakeshop chain you have never seen before.
**60 minutes · open notes (your own notebooks and code) · **

Five sections. Every section ends with a printed **EVIDENCE** result and a one-sentence **READING**.
Nothing changes silently — print your counts. A correct number with no reading earns nothing.

In [1]:
# SETUP — run first. Your 60 minutes start here.
from datetime import datetime
from zoneinfo import ZoneInfo
STUDENT_NAME = "Sanchez, Eliah Therrien"      # Surname, First Name
NOTEBOOK = "A"
START_TIME = datetime.now(ZoneInfo("Asia/Manila"))
assert STUDENT_NAME.strip(), "Type your name before anything else."
print("Examinee:", STUDENT_NAME, "| Notebook", NOTEBOOK, "| started", START_TIME.strftime("%Y-%m-%d %H:%M"))

Examinee: Sanchez, Eliah Therrien | Notebook A | started 2026-10-03 15:59


In [2]:
# LOAD — paste the four FILE_IDs from the access sheet, then run.
import pandas as pd, numpy as np
FILE_ID_ORDERS   = "1zLamAc57ODw-QKtKCjRDrae3IlEqij_b"
FILE_ID_BRANCHES = "1gcXPflDscqrL5gbAcfhwBRRVJoT6vBxf"
FILE_ID_CALENDAR = "1WWBZZIg4m2vtEsz1vw0SxTH6_qOV7Fru"
FILE_ID_PREDS    = "1NkouL2RnS6ygzVGm1xbJ_dHV8KRnC1oO"
def _url(fid): return f"https://drive.google.com/uc?export=download&id={fid}"
orders   = pd.read_csv(_url(FILE_ID_ORDERS), dtype={"AmountPHP": str}, keep_default_na=False)
branches = pd.read_csv(_url(FILE_ID_BRANCHES))
calendar = pd.read_csv(_url(FILE_ID_CALENDAR), parse_dates=["Date"])
preds    = pd.read_csv(_url(FILE_ID_PREDS), parse_dates=["Date"])
print(orders.shape, branches.shape, calendar.shape, preds.shape)
orders.head(3)

(42028, 7) (6, 5) (608, 5) (552, 3)


,OrderID,Date,BranchID,ItemsSold,Channel,Payment,AmountPHP
0,BM108937,2025-12-16,P03,4,Store,Cash,67.33
1,BM101926,2025-05-22,P05,3,GrabFood,Cash,176.22
2,BM120165,2026-05-27,P05,4,Store,GCash,80.41


---
## §1 · Prepare, then split — the calendar does the splitting  *(10 points)*

The file carries **five known defect families**: duplicated orders, dirty BranchID variants,
text-trapped amounts, blank dates, and orphan branch rows. Four are repaired for you below —
run and check the printed counts. **Your spotlight defect: dirty BranchID variants.**
Six misspelled versions of real branch codes are hiding in BranchID, and a code the branch table has never heard of. Find every variant, count the rows, repair them to the real codes — then remove the rows whose code is not in branches.csv, counted and reasoned.

Then split by the calendar: **training = everything before 2026-06-01 · holdout = 2026-06-01 onward.**

**EVIDENCE:** the count for your spotlight repair, the clean row count, and the train / holdout branch-day counts.
**READING:** one sentence — why does the calendar make this split, and not `sample()`?

In [3]:
# GIVEN repairs — four of the five families. Run and check each count.
work = orders.copy()
print("raw rows:", len(work))

work = work.drop_duplicates(subset=["OrderID"])
print("after dedupe:", len(work))

work["AmountPHP"] = work["AmountPHP"].str.replace(",", "").astype(float)
print("AmountPHP dtype:", work["AmountPHP"].dtype)

work = work[work["Date"] != ""]
work["Date"] = pd.to_datetime(work["Date"])
print("after date repair:", len(work))

raw rows: 42028
after dedupe: 42000
AmountPHP dtype: float64
after date repair: 41984


In [4]:
# YOUR spotlight repair — dirty BranchID variants.
# Investigate, count, repair. Print the count BEFORE you repair it.

# The six dirty spellings and the real branch codes they represent.
branch_fixes = {
    "P 05": "P05",
    "P-02": "P02",
    "P03.": "P03",
    "P06 ": "P06",
    "p01": "P01",
    "p04": "P04",
}

# Count the spotlight rows BEFORE repairing them.
dirty_mask = work["BranchID"].isin(branch_fixes)
print("EVIDENCE dirty BranchID rows repaired:", int(dirty_mask.sum()))
print("Dirty variants:")
print(work.loc[dirty_mask, "BranchID"].value_counts().sort_index())

# Repair the six variants.
work["BranchID"] = work["BranchID"].replace(branch_fixes)

# Anything still not found in branches.csv is an orphan.
valid_branch_ids = set(branches["BranchID"].astype(str))
orphan_mask = ~work["BranchID"].astype(str).isin(valid_branch_ids)
print("EVIDENCE orphan rows removed:", int(orphan_mask.sum()))
print("Orphan BranchIDs:")
print(work.loc[orphan_mask, "BranchID"].value_counts().sort_index())

# Remove the orphan rows.
work = work.loc[~orphan_mask].copy()


EVIDENCE dirty BranchID rows repaired: 46
Dirty variants:
BranchID
P 05    7
P-02    8
P03.    8
P06     7
p01     8
p04     8
Name: count, dtype: int64
EVIDENCE orphan rows removed: 21
Orphan BranchIDs:
BranchID
P07    21
Name: count, dtype: int64


In [5]:
# EVIDENCE §1 — run after your repair. All five families must be repaired by this point.
clean = work.copy()
print("EVIDENCE clean rows:", len(clean))
bd = clean.groupby(["BranchID","Date"], as_index=False)["AmountPHP"].sum().rename(columns={"AmountPHP":"Revenue"})
SPLIT = pd.Timestamp("2026-06-01")
train, hold = bd[bd["Date"] < SPLIT], bd[bd["Date"] >= SPLIT]
print("EVIDENCE branch-days:", len(bd), "| train:", len(train), "| holdout:", len(hold))

EVIDENCE clean rows: 41963
EVIDENCE branch-days: 3647 | train: 3095 | holdout: 552


**READING §1:**  
The calendar makes the split because it preserves the time order of the data, using earlier branch-days for training and later branch-days for holdout, whereas sample() could mix future observations into training.


---
## §2 · Features and the leakage test  *(15 points)*

The test is one question: **was this knowable at the moment the decision would be taken?**
The decision: *predict a branch-day's revenue, the evening before.*

**(a) Build your assigned feature — `IsWeekend`.** Join the calendar and build IsWeekend for each branch-day. EVIDENCE: mean revenue weekend vs weekday, and the premium in percent.

**(b) The candidate table.** Classify each candidate **KEEP** or **LEAK**, with a calendar reason.

| # | Candidate feature | What the column is |
|---|---|---|
| 1 | DayOfWeek of the day being predicted | calendar fact, fixed in advance |
| 2 | PromoPlanned flag for the day being predicted | planned by head office weeks ahead |
| 3 | The same branch's revenue 7 days earlier | already finished and recorded |
| 4 | ItemsSold on the day being predicted | counted only as the day happens |
| 5 | Month-to-date revenue including the day being predicted | contains the target day inside the sum |
| 6 | Refunds processed on the day being predicted | processed during and after the day |


In [6]:
# (a) Build IsWeekend and print its EVIDENCE.
# Join the branch-day table to the supplied calendar.
bd_cal = bd.merge(calendar, on="Date", how="left", validate="many_to_one")

# Use the calendar's DayOfWeek when available; handle common numeric/string encodings.
dow = bd_cal["DayOfWeek"]
if pd.api.types.is_numeric_dtype(dow):
    # pandas-style Monday=0 ... Sunday=6
    bd_cal["IsWeekend"] = dow.isin([5, 6])
else:
    bd_cal["IsWeekend"] = dow.astype(str).str.strip().str.lower().isin(
        ["saturday", "sunday", "sat", "sun"]
    )

weekend_mean = bd_cal.loc[bd_cal["IsWeekend"], "Revenue"].mean()
weekday_mean = bd_cal.loc[~bd_cal["IsWeekend"], "Revenue"].mean()
premium_pct = (weekend_mean - weekday_mean) / weekday_mean * 100

print("EVIDENCE weekend mean revenue:", round(weekend_mean, 2))
print("EVIDENCE weekday mean revenue:", round(weekday_mean, 2))
print("EVIDENCE weekend premium (%):", round(premium_pct, 2))


EVIDENCE weekend mean revenue: 1965.98
EVIDENCE weekday mean revenue: 1386.16
EVIDENCE weekend premium (%): 41.83


**(b) Verdicts — fill every row:**

| # | KEEP or LEAK | Calendar reason (one clause) |
|---|---|---|
| 1 | KEEP | known from the calendar before the predicted day |
| 2 | KEEP | planned before the prediction date |
| 3 | KEEP | the earlier branch-day has already finished and been recorded |
| 4 | LEAK | only known as the predicted day occurs |
| 5 | LEAK | the sum includes the target day itself |
| 6 | LEAK | processed during or after the predicted day |

**READING §2:** *(one sentence — the feature test, in your own words)*

A feature is usable only if its value would already be knowable at the evening before the branch-day being predicted.


---
## §3 · Baseline first — you are not allowed to beat nothing  *(10 points)*

Before looking at any model: compute the **naive-7 baseline** — predict each holdout branch-day
with the same branch's revenue **7 days earlier** — and score it with **MAE** on both of your assigned branches.
Your assigned slice: **P01 and P03**.

**EVIDENCE:** the baseline MAE, printed per assigned slice.
**READING:** one sentence — why is this baseline computed *before* the model is examined?

In [7]:
# §3 — naive-7 baseline on the holdout.
ASSIGNED = ['P01', 'P03']
bdi = bd.set_index(["BranchID","Date"])["Revenue"]
ho = hold.copy()
ho["Naive7"] = [bdi.get((r.BranchID, r.Date - pd.Timedelta(days=7)), np.nan) for r in ho.itertuples()]
ho = ho.dropna(subset=["Naive7"])
def mae(a, f): return float(np.mean(np.abs(a - f)))
for tag in ASSIGNED:
    g = ho if tag == "ALL" else ho[ho["BranchID"] == tag]
    print(f"EVIDENCE baseline MAE [{tag}]:", round(mae(g["Revenue"], g["Naive7"]), 2))

EVIDENCE baseline MAE [P01]: 1140.51
EVIDENCE baseline MAE [P03]: 738.82


**READING §3:**

> The naive-7 baseline is computed first so the model has a fixed, pre-existing benchmark for judging whether its prediction error is lower.


---
## §4 · The model must earn it  *(15 points)*

Head office bought **“Model M,”** a forecasting service. Its predictions for every holdout
branch-day are in `preds`. Score Model M with MAE on your assigned slice —
**P01 and P03** — beside your §3 baseline, and compute the ratio
**MAE(Model M) ÷ MAE(naive-7)**. Below 1.0 = the model earns its keep on that slice.

**EVIDENCE:** both MAEs and the ratio, per assigned slice, with **WINS / LOSES** printed.
**READING:** one sentence per slice — what the ratio says, in pesos-and-decisions language.

In [8]:
# §4 — Model M vs your baseline.
ev = ho.merge(preds, on=["BranchID","Date"], how="left").dropna(subset=["ModelM_PredictedRevenue"])
for tag in ASSIGNED:
    g = ev if tag == "ALL" else ev[ev["BranchID"] == tag]
    m_b = mae(g["Revenue"], g["Naive7"]); m_m = mae(g["Revenue"], g["ModelM_PredictedRevenue"])
    r = m_m / m_b
    print(f"EVIDENCE [{tag}] baseline {m_b:,.2f} | Model M {m_m:,.2f} | ratio {r:.3f} |", "MODEL WINS" if r < 1 else "MODEL LOSES")

EVIDENCE [P01] baseline 1,140.51 | Model M 883.76 | ratio 0.775 | MODEL WINS
EVIDENCE [P03] baseline 738.82 | Model M 849.89 | ratio 1.150 | MODEL LOSES


**READING §4:**

> For each branch, the ratio compares Model M's MAE with the naive-7 MAE: a ratio below 1 means Model M has lower error, while a ratio at or above 1 means it does not.


---
## §5 · Judgment — the memo line  *(5 points)*

Head office asks one question: **“Do we roll Model M out to every branch?”**

Write your verdict in **three sentences maximum**, using only numbers your own notebook printed:
what you recommend, which evidence carries it, and what you would check next.
A verdict your own evidence does not support earns nothing.

**MEMO:**

>

In [10]:
# FINISH — run last.
from datetime import datetime
from zoneinfo import ZoneInfo
END_TIME = datetime.now(ZoneInfo("Asia/Manila"))
def _surname(name):
    if "," in name: return name.split(",")[0].strip().upper()
    parts = [p for p in name.split() if p.strip(".")]
    return (parts[-1] if parts else "SURNAME").upper()
mins = (END_TIME - START_TIME).total_seconds() / 60
print(f"Examinee: {STUDENT_NAME} | Notebook {NOTEBOOK} | elapsed {mins:.0f} min")
print(f"Save as: BDAT05_Midterm_Part2A_{NOTEBOOK}_{_surname(STUDENT_NAME)}.ipynb and upload to your midterm folder.")

Examinee: Sanchez, Eliah Therrien | Notebook A | elapsed 5 min
Save as: BDAT05_Midterm_Part2A_A_SANCHEZ.ipynb and upload to your midterm folder.
